# Validate MODIS (MCDWD) pixel-code legend from the LANCE GeoTIFFs

**Goal.** Same exercise we just ran for VIIRS, now for MODIS. We:

1. List the legend NASA documents in the bundled **MCDWD/VCDWD User Guide Rev F** Table 7
   (`docs/mcdwd_vcdwd_userguide_revF.md`).
2. Pull a raw GeoTIFF from each MODIS backend Atlantis supports, read its embedded TIFF tags,
   sample the actual pixel codes, and cross-check against the user guide + the Atlantis decoder.
3. Flag any disagreement programmatically.

Atlantis declares two MODIS backends in `src/atlantis/fetchers/modis/backend.py`:

| Backend id      | Class                | Source URL pattern                                                                                | Auth                        |
| --------------- | -------------------- | ------------------------------------------------------------------------------------------------- | --------------------------- |
| `lance_geotiff` | `LanceGeotiffBackend`| `https://nrt3.modaps.eosdis.nasa.gov/archive/allData/61/MCDWD_L3_<COMPOSITE>_NRT/YYYY/DDD/`       | Earthdata bearer            |
| `laads_hdf4`    | `LaadsHdf4Backend`   | `https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/61/MCDWD_L3_<COMPOSITE>/...`              | Earthdata bearer + URS app pre-auth |

**This notebook focuses on `lance_geotiff` only.** It needs only `EARTHDATA_TOKEN` (which the
repo's `.env` already provides), it returns plain single-band GeoTIFFs that `/vsicurl/` can
range-read, and it carries the same pixel encoding as the HDF4 archive — what we learn here
applies verbatim to the LAADS HDF4 product.

> Internet access is required. No bytes are written to disk; we stream small windows.

## 1. Setup

Resolve the workspace root from this notebook's location, load `.env`, put the `src/` directory
on `sys.path`, and ensure GDAL forwards the Earthdata bearer token on every `/vsicurl/` request.

In [ ]:
from __future__ import annotations

import os
import sys
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
candidate = NOTEBOOK_DIR
while candidate != candidate.parent and not (candidate / 'pyproject.toml').exists():
    candidate = candidate.parent
REPO_ROOT = candidate
SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Load .env so EARTHDATA_TOKEN is available without leaking it to the notebook output.
from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env', override=False)

TOKEN_OK = bool(os.environ.get('EARTHDATA_TOKEN'))
print(f'REPO_ROOT          = {REPO_ROOT}')
print(f'notebook           = {NOTEBOOK_DIR.relative_to(REPO_ROOT)}')
print(f'EARTHDATA_TOKEN ok = {TOKEN_OK}')

os.environ.setdefault('GDAL_HTTP_TIMEOUT', '30')
os.environ.setdefault('GDAL_HTTP_MAX_RETRY', '3')
os.environ.setdefault('CPL_VSIL_CURL_CHUNK_SIZE', '524288')  # 512 KiB

In [ ]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window, from_bounds

from atlantis.fetchers.modis.backend import (
    LanceGeotiffBackend,
    ModisListingEntry,
    earthdata_auth_headers,
    get_backend,
    list_backends,
)
from atlantis.fetchers.modis.processor import (
    INSUFFICIENT_DATA_CODE,
    NO_WATER_CODE,
    RECURRING_FLOOD_CODE,
    SURFACE_WATER_CODE,
    UNUSUAL_FLOOD_CODE,
    modis_tiles_for_bbox,
)

ACTIVE_BACKENDS = ['lance_geotiff']  # laads_hdf4 needs URS app pre-auth; out of scope here.
print('Atlantis MODIS backends declared :', list_backends())
print('Backends used in this notebook   :', ACTIVE_BACKENDS)

## 2. Reference legends, side-by-side

Unlike VIIRS, MODIS has **one canonical legend** — NASA's MCDWD/VCDWD User Guide Rev F Table 7.
We capture it here as the source of truth, then snapshot what Atlantis and our docs claim so we
can diff against the live raster later.

- **NASA legend** — `docs/mcdwd_vcdwd_userguide_revF.md` Table 7 (Release 1.1, Dec 2025).
- **Atlantis decoder** — live constants pulled from `atlantis.fetchers.modis.processor`.
- **Docs/notes snapshot** — what `docs/modis/overview.md` claims, kept static here for
  drift detection.

In [ ]:
# --- Reference 1: NASA MCDWD/VCDWD User Guide Rev F, Table 7 (canonical) ---
NASA_LEGEND: dict[int, str] = {
    0:   'No water',
    1:   'Surface water (matching reference water)',
    2:   'Recurring flood',
    3:   'Flood (unusual)',
    255: 'Insufficient data',
}

# --- Reference 2: live Atlantis decoder constants ---
ATLANTIS_DECODER = {
    'NO_WATER_CODE':          NO_WATER_CODE,
    'SURFACE_WATER_CODE':     SURFACE_WATER_CODE,
    'RECURRING_FLOOD_CODE':   RECURRING_FLOOD_CODE,
    'UNUSUAL_FLOOD_CODE':     UNUSUAL_FLOOD_CODE,
    'INSUFFICIENT_DATA_CODE': INSUFFICIENT_DATA_CODE,
}
for k, v in ATLANTIS_DECODER.items():
    print(f'{k:>24}  {v}')

# --- Reference 3: docs/modis/overview.md "quick reference" snapshot ---
DOCS_OVERVIEW_LEGEND: dict[int, str] = {
    0:   'No water',
    1:   'Surface / reference water',
    2:   'Recurring flood',
    3:   'Unusual flood',
    255: 'Insufficient data',
}

In [ ]:
def atlantis_meaning(code: int) -> str:
    """Show what `_classify_pixels` actually does with each code."""
    if code == NO_WATER_CODE:
        return 'flood_fraction=0, quality=1 (background)'
    if code == SURFACE_WATER_CODE:
        return 'permanent_water=1, quality=1'
    if code == RECURRING_FLOOD_CODE:
        return 'recurring_flood=1, quality=1'
    if code == UNUSUAL_FLOOD_CODE:
        return 'flood_fraction=1.0, quality=1'
    if code == INSUFFICIENT_DATA_CODE:
        return 'quality_mask=0 (sentinel/nodata)'
    return '— (unknown to decoder; passes through)'

all_codes = sorted(set(NASA_LEGEND) | set(DOCS_OVERVIEW_LEGEND))
legend_df = pd.DataFrame(
    [
        {
            'code':              c,
            'NASA Table 7':      NASA_LEGEND.get(c, ''),
            'docs/modis/overview.md': DOCS_OVERVIEW_LEGEND.get(c, ''),
            'Atlantis decoder':  atlantis_meaning(c),
        }
        for c in all_codes
    ]
).set_index('code')
legend_df

All three references already agree on the five canonical codes. This is fundamentally different
from VIIRS — where ATBD, public TIFF tags, and Atlantis disagreed in several places.

What's left to validate, then, is:

1. Does the live LANCE GeoTIFF actually contain only `{0, 1, 2, 3, 255}` and nothing else?
2. Does its embedded metadata (TIFF tags, GDAL metadata) carry an authoritative legend like NOAA
   does for VIIRS, or is it silent?
3. Are any **other** subtle issues hiding in the Atlantis processor / docs (e.g., the M1 finding
   in `tmp/source_layers_comparison.md` about `cloud_fraction` overstating `255`)?

## 3. Pick a date and AOI for the probe

LANCE NRT carries a rolling ~1-week window. We try the last few days in order, falling back if a
particular date hasn't been published yet. For the AOI we use the **Mekong delta** (h27v07,
h27v08) — it has reliable recurring flood + surface water signal year-round, so we'll almost
certainly see codes 1 and 2 alongside 0, and frequently 3.

In [ ]:
MEKONG_BBOX = (104.0, 9.0, 107.0, 11.5)  # west, south, east, north (EPSG:4326)
PROBE_COMPOSITE = 'F2'  # 2-Day composite — most flood signal per pixel

tiles_hv = modis_tiles_for_bbox(MEKONG_BBOX)
print(f'Mekong delta bbox → MODIS tiles (h,v): {tiles_hv}')
PROBE_H, PROBE_V = tiles_hv[0]
print(f'Using h{PROBE_H:02d}v{PROBE_V:02d} for the rest of this notebook.')

# LANCE keeps roughly the past 7 days. Today (UTC) may not yet have a product;
# yesterday and the day before are the safest probes.
today_utc = datetime.now(timezone.utc).date()
PROBE_DATE_CANDIDATES = [
    datetime.combine(today_utc - timedelta(days=2), datetime.min.time(), tzinfo=timezone.utc),
    datetime.combine(today_utc - timedelta(days=3), datetime.min.time(), tzinfo=timezone.utc),
    datetime.combine(today_utc - timedelta(days=4), datetime.min.time(), tzinfo=timezone.utc),
    datetime.combine(today_utc - timedelta(days=1), datetime.min.time(), tzinfo=timezone.utc),
]
print('Probe date candidates (UTC):', [d.date().isoformat() for d in PROBE_DATE_CANDIDATES])

## 4. Resolve one remote raw TIFF per backend

Same recipe as the VIIRS notebook, just using MODIS-shaped APIs:

1. `get_listing_location(...)` — build the per-date archive prefix.
2. `get_directory_listing(...)` — hit the LANCE JSON listing API.
3. `find_remote_filename(h, v, composite, entries)` — pick the file for our tile.
4. `build_result_url(...)` — assemble the GET-able URL.

We **do not download** anything; we keep the URL and read it via `/vsicurl/`.

In [ ]:
BACKEND_BASE_URLS = {
    'lance_geotiff': 'https://nrt3.modaps.eosdis.nasa.gov',
}
TIMEOUT = 30

def resolve_remote_url(backend_id: str) -> tuple[str | None, datetime | None]:
    backend = get_backend(backend_id)
    base_url = BACKEND_BASE_URLS[backend_id]
    headers = earthdata_auth_headers()  # bearer token from EARTHDATA_TOKEN
    for dt in PROBE_DATE_CANDIDATES:
        loc = backend.get_listing_location(base_url, dt, PROBE_COMPOSITE)
        try:
            entries = backend.get_directory_listing(base_url, loc, TIMEOUT, headers=headers)
        except Exception as exc:  # noqa: BLE001
            print(f'[{backend_id}] {dt.date()} listing failed: {exc}')
            continue
        if not entries:
            print(f'[{backend_id}] {dt.date()} listing empty (date not yet published?)')
            continue
        entry = backend.find_remote_filename(PROBE_H, PROBE_V, PROBE_COMPOSITE, entries)
        if entry is None:
            print(
                f'[{backend_id}] {dt.date()} no h{PROBE_H:02d}v{PROBE_V:02d} '
                f'match for composite {PROBE_COMPOSITE} in {len(entries)} entries'
            )
            continue
        url = backend.build_result_url(base_url, loc, entry)
        return url, dt
    return None, None

backend_urls: dict[str, str | None] = {}
backend_dates: dict[str, datetime | None] = {}
for bid in ACTIVE_BACKENDS:
    url, dt = resolve_remote_url(bid)
    backend_urls[bid] = url
    backend_dates[bid] = dt
    print(f'\n--- {bid} ---')
    print(f'  date used    : {dt.date() if dt else "—"}')
    print(f'  resolved URL : {url}')

## 5. Read embedded metadata from each remote TIFF

MODIS LANCE products are single-band uint8 GeoTIFFs. Any documentation of the pixel encoding
should live in the global TIFF tags or per-band tags. We open the remote file with the bearer
token forwarded through GDAL, then dump everything.

In [ ]:
def vsicurl(url: str) -> str:
    return url if url.startswith('/vsicurl/') else f'/vsicurl/{url}'

# Tags whose full content we want to see verbatim (no preview truncation), because
# they may carry the pixel-code legend.
FULL_TAGS = {'description', 'long_name', 'valid_range', '_FillValue'}

metadata_tags: dict[str, dict[str, object]] = {}
token = os.environ.get('EARTHDATA_TOKEN', '').strip()
gdal_env = {'GDAL_HTTP_HEADERS': f'Authorization: Bearer {token}'} if token else {}

for bid, url in backend_urls.items():
    if not url:
        print(f'[{bid}] skipped — no URL')
        continue
    try:
        with rasterio.Env(**gdal_env):
            with rasterio.open(vsicurl(url)) as ds:
                metadata_tags[bid] = {
                    'driver':       ds.driver,
                    'dtype':        ds.dtypes[0],
                    'count':        ds.count,
                    'width':        ds.width,
                    'height':       ds.height,
                    'crs':          str(ds.crs),
                    'transform':    tuple(ds.transform)[:6],
                    'nodata':       ds.nodata,
                    'tags':         ds.tags(),
                    'band1_tags':   ds.tags(1),
                    'descriptions': ds.descriptions,
                }
    except Exception as exc:  # noqa: BLE001
        print(f'[{bid}] rasterio open failed: {exc}')
        metadata_tags[bid] = {'error': str(exc)}

for bid, meta in metadata_tags.items():
    print(f'\n===== {bid} =====')
    for k, v in meta.items():
        if k in ('tags', 'band1_tags') and isinstance(v, dict):
            print(f'  {k}:')
            for tk, tv in v.items():
                if tk in FULL_TAGS:
                    # Print full, multi-line tag verbatim.
                    print(f'    - {tk}: |')
                    for line in str(tv).splitlines():
                        print(f'        {line}')
                else:
                    preview = tv if len(str(tv)) < 200 else str(tv)[:200] + ' …'
                    print(f'    - {tk}: {preview}')
        else:
            print(f'  {k}: {v}')

## 6. Extract the embedded pixel-code legend from the `description` tag

The LANCE GeoTIFF embeds a complete legend in its GDAL `description` tag, under a `Flag values:`
header. We parse it here so it can join the cross-check just like the NOAA VIIRS
`WaterDetection#TypeDescription` tag did. We also pull out the per-file processing notes
(cloud / terrain shadow / HAND / threshold) — they're directly relevant to the M1 finding.

In [ ]:
import re

# Matches lines like '    255 : Insufficient data' inside the description tag.
_FLAG_PATTERN = re.compile(r'^\s*(\d{1,3})\s*:\s*(.+?)\s*$', re.MULTILINE)

def parse_embedded_legend(meta: dict[str, object]) -> dict[int, str]:
    """Pull the pixel-code legend out of the LANCE TIFF `description` tag."""
    tags = meta.get('tags')
    if not isinstance(tags, dict):
        return {}
    description = tags.get('description')
    if not isinstance(description, str):
        return {}
    # The legend block starts at 'Flag values:'.
    if 'Flag values:' not in description:
        return {}
    block = description.split('Flag values:', 1)[1]
    legend: dict[int, str] = {}
    for m in _FLAG_PATTERN.finditer(block):
        try:
            code = int(m.group(1))
        except ValueError:
            continue
        if 0 <= code <= 255:
            legend[code] = m.group(2).strip()
    return dict(sorted(legend.items()))

# Also capture the per-file processing notes (cloud / terrain shadow / HAND).
def parse_masking_notes(meta: dict[str, object]) -> list[str]:
    tags = meta.get('tags')
    if not isinstance(tags, dict):
        return []
    description = tags.get('description')
    if not isinstance(description, str):
        return []
    notes: list[str] = []
    for line in description.splitlines():
        stripped = line.strip()
        if any(kw in stripped.lower() for kw in ('cloud', 'terrain shadow', 'hand ', 'threshold')):
            notes.append(stripped)
    return notes

embedded_legends: dict[str, dict[int, str]] = {}
masking_notes: dict[str, list[str]] = {}
for bid, meta in metadata_tags.items():
    if meta.get('error'):
        embedded_legends[bid] = {}
        masking_notes[bid] = []
        continue
    embedded_legends[bid] = parse_embedded_legend(meta)
    masking_notes[bid] = parse_masking_notes(meta)
    print(f'\n[{bid}] embedded legend ({len(embedded_legends[bid])} codes):')
    for c, lbl in embedded_legends[bid].items():
        print(f'  {c:>3}: {lbl}')
    if masking_notes[bid]:
        print(f'[{bid}] embedded processing notes:')
        for n in masking_notes[bid]:
            print(f'  • {n}')

## 7. Sample actual pixel values

Header tells us what the file *should* contain; sampling tells us what's *actually* there.
We read a window over the Mekong delta bbox.

In [ ]:
def sample_codes(url: str, bbox: tuple[float, float, float, float]) -> pd.Series:
    with rasterio.Env(**gdal_env):
        with rasterio.open(vsicurl(url)) as ds:
            win = from_bounds(*bbox, transform=ds.transform).round_offsets().round_lengths()
            win = win.intersection(Window(0, 0, ds.width, ds.height))
            arr = ds.read(1, window=win)
    values, counts = np.unique(arr, return_counts=True)
    s = pd.Series(counts, index=values, name='count').sort_index()
    s.index.name = 'code'
    return s

code_samples: dict[str, pd.Series] = {}
for bid, url in backend_urls.items():
    if not url:
        continue
    try:
        code_samples[bid] = sample_codes(url, MEKONG_BBOX)
        print(f'[{bid}] {len(code_samples[bid])} distinct codes in Mekong bbox window')
    except Exception as exc:  # noqa: BLE001
        print(f'[{bid}] sampling failed: {exc}')

if code_samples:
    samples_df = pd.concat(code_samples, axis=1).fillna(0).astype('int64')
    samples_df.columns = [f'{bid}__count' for bid in samples_df.columns]
    for bid in code_samples:
        col = f'{bid}__count'
        samples_df[f'{bid}__pct'] = (samples_df[col] / samples_df[col].sum() * 100).round(2)
    display(samples_df)

## 8. Master cross-check table

One row per code that appears in any legend or any sampled raster. The embedded LANCE legend is
the authoritative reference here — every other column either echoes it (good) or drifts from it
(actionable).

In [ ]:
code_universe: set[int] = set(NASA_LEGEND) | set(DOCS_OVERVIEW_LEGEND)
for legend in embedded_legends.values():
    code_universe |= set(legend)
for s in code_samples.values():
    code_universe |= {int(c) for c in s.index}

rows = []
for code in sorted(code_universe):
    row = {
        'code':                  code,
        'NASA Table 7':          NASA_LEGEND.get(code, ''),
        'docs/modis/overview':   DOCS_OVERVIEW_LEGEND.get(code, ''),
        'Atlantis decoder':      atlantis_meaning(code),
    }
    for bid in ACTIVE_BACKENDS:
        row[f'embedded [{bid}]'] = embedded_legends.get(bid, {}).get(code, '')
        s = code_samples.get(bid)
        row[f'count [{bid}]']    = int(s.get(code, 0)) if s is not None else 0
    rows.append(row)

crosscheck_df = pd.DataFrame(rows).set_index('code')
with pd.option_context('display.max_colwidth', 80):
    display(crosscheck_df)

## 9. Programmatic findings

Same shape as the VIIRS notebook. The embedded LANCE legend (parsed in section 6) is treated as
the authoritative reference; every other column is checked against it. We also surface the
per-file processing notes because they're the data the M1 finding hinges on.

In [ ]:
findings: list[str] = []

# The embedded LANCE legend is the source of truth here. Use the noaa-style helper.
def _embedded(code: int) -> str:
    for leg in embedded_legends.values():
        if code in leg:
            return leg[code]
    return ''

# F1 — Live raster contains only documented codes?
unknown_codes: set[int] = set()
known = set(NASA_LEGEND) | set().union(*(set(leg) for leg in embedded_legends.values()))
for s in code_samples.values():
    unknown_codes |= {int(c) for c in s.index if int(c) not in known}
if unknown_codes:
    findings.append(
        f'F1 [OPEN]: Live raster contains codes outside NASA Table 7 AND outside the embedded '
        f'LANCE legend: {sorted(unknown_codes)}. Atlantis falls through to the "unknown" branch.'
    )
else:
    findings.append('F1 [OK]: Live raster contains only documented codes (NASA Table 7 = embedded LANCE legend).')

# F2 — NASA Table 7 and the embedded LANCE legend agree?
nasa_keys = set(NASA_LEGEND)
embed_keys = set().union(*(set(leg) for leg in embedded_legends.values()))
if nasa_keys != embed_keys:
    findings.append(
        f'F2 [OPEN]: NASA Table 7 codes {sorted(nasa_keys)} differ from the embedded LANCE legend codes '
        f'{sorted(embed_keys)}. Reconcile docs.'
    )
else:
    findings.append('F2 [OK]: NASA Table 7 codes match the embedded LANCE legend exactly.')

# F3 — Atlantis decoder constants match the embedded LANCE legend semantically.
decoder_to_label = {
    NO_WATER_CODE:          'No water',
    SURFACE_WATER_CODE:     'water',          # "Water (matches standard reference water)"
    RECURRING_FLOOD_CODE:   'Recurring flood',
    UNUSUAL_FLOOD_CODE:     'Flood',
    INSUFFICIENT_DATA_CODE: 'Insufficient',
}
mismatches = []
for code, kw in decoder_to_label.items():
    label = _embedded(code).lower()
    if label and kw.lower() not in label:
        mismatches.append((code, kw, _embedded(code)))
if mismatches:
    findings.append(f'F3 [OPEN]: Atlantis decoder constants disagree with embedded LANCE legend: {mismatches}.')
else:
    findings.append('F3 [OK]: Atlantis decoder constants are aligned with the embedded LANCE legend.')

# F4 / M1 — `cloud_fraction` docstring previously claimed 255 = cloud + terrain shadow + HAND
# unconditionally. After the M1 fix the docstring acknowledges the field is a misnomer and
# describes the per-composite reality (cloud removal depends on the composite). We treat the
# presence of either "misnomer" or "fraction of insufficient-data" as the signal that the fix
# has landed.
from atlantis.fetchers.modis.processor import ProcessedTile
tile_doc = (ProcessedTile.__doc__ or '').lower()
notes_blob = ' | '.join(' '.join(masking_notes.get(bid, [])).lower() for bid in ACTIVE_BACKENDS)
docstring_fixed = ('misnomer' in tile_doc) or ('fraction of insufficient-data' in tile_doc) or ('insufficient-data pixels' in tile_doc)
file_says_no_cloud_removed = 'no clouds removed' in notes_blob

if docstring_fixed:
    findings.append(
        'F4 [RESOLVED]: `ProcessedTile.cloud_fraction` docstring now flags the field as a '
        'misnomer and describes per-composite semantics (the embedded LANCE description is '
        'authoritative). Matches the M1 fix in src/atlantis/fetchers/modis/processor.py.'
    )
elif file_says_no_cloud_removed and 'cloud' in tile_doc and 'terrain shadow' in tile_doc and 'hand' in tile_doc:
    findings.append(
        'F4 [OPEN]: `ProcessedTile.cloud_fraction` docstring claims 255 includes "cloud + '
        'terrain shadow + HAND", but the embedded LANCE description for the F2 composite says '
        '"no clouds removed". The docstring overstates what 255 covers for F2/F3.'
    )

# F5 — Wording drift in docs/modis/overview.md vs NASA Table 7.
doc_diffs = []
for code, nasa_label in NASA_LEGEND.items():
    docs_label = DOCS_OVERVIEW_LEGEND.get(code, '')
    if docs_label and docs_label != nasa_label:
        doc_diffs.append((code, nasa_label, docs_label))
if doc_diffs:
    findings.append(
        f'F5 [OPEN/cosmetic]: Wording drift in docs/modis/overview.md vs NASA Table 7 '
        f'(same codes, slightly different phrasing): {doc_diffs}.'
    )

for f in findings:
    print('•', f)
    print()

## 10. Visual sanity check

Bar chart of the actual code distribution inside the Mekong delta bbox window.

In [ ]:
import matplotlib.pyplot as plt

bid = ACTIVE_BACKENDS[0]
if bid in code_samples:
    s = code_samples[bid].sort_values(ascending=False)
    labels = [f'{int(c)} {NASA_LEGEND.get(int(c), "?")}' for c in s.index][::-1]
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.barh(labels, s.values[::-1])
    ax.set_xlabel(f'pixel count (Mekong bbox window, [{bid}])')
    ax.set_title(f'MODIS code distribution — {bid} on {backend_dates[bid].date()}')
    for i, v in enumerate(s.values[::-1]):
        ax.text(v, i, f' {v:,}', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print('No samples — cannot plot.')

## 11. Summary

Where VIIRS had three independent references that disagreed (and Atlantis followed the wrong
one in three places), MODIS is in much better shape:

1. **The LANCE GeoTIFF embeds the canonical legend in its `description` tag** (`Flag values:`
   block), alongside per-composite processing notes (cloud / terrain shadow / HAND / threshold).
   This is the source of truth we trust here — it travels with every file.
2. **NASA's MCDWD/VCDWD User Guide Rev F Table 7** (bundled at
   `docs/mcdwd_vcdwd_userguide_revF.md`) matches the embedded legend exactly: codes 0/1/2/3/255
   with the same wording.
3. **Atlantis' decoder constants line up exactly** with that legend
   (`NO_WATER_CODE = 0`, `SURFACE_WATER_CODE = 1`, `RECURRING_FLOOD_CODE = 2`,
   `UNUSUAL_FLOOD_CODE = 3`, `INSUFFICIENT_DATA_CODE = 255`).

### Status of each finding

| # | Topic | Status | Notes |
| --- | --- | --- | --- |
| F1 | Live raster contains only documented codes | **OK** | Verified for the F2 composite over the Mekong delta. |
| F2 | NASA Table 7 matches the embedded LANCE legend | **OK** | Same five codes with consistent semantics. |
| F3 | Atlantis decoder constants align with the embedded LANCE legend | **OK** | All five constants point at the correct labels. |
| F4 (M1) | `cloud_fraction` docstring overstated what code 255 covers per composite | **Resolved** | Docstring in [src/atlantis/fetchers/modis/processor.py](src/atlantis/fetchers/modis/processor.py) tightened — now flags the field as a misnomer and points at the embedded `description` tag as authoritative. |
| F5 | Cosmetic wording drift in `docs/modis/overview.md` vs NASA Table 7 | **Open (cosmetic)** | Same codes, slightly different phrasing (e.g. `Surface water (matching reference water)` vs `Surface / reference water`). Harmless. |

### Why this matters

Even after the docstring fix, the **field is still named `cloud_fraction`** even though it
numerically measures the fraction of code-255 ("Insufficient data") pixels — which is not
strictly cloud cover, especially for F2/F3 composites that do not remove clouds at all. Renaming
would cascade through `TileMetadata.cloud_fraction`, every consumer, and the CLI output. The
docstring tightening is the pragmatic fix; renaming is a separate refactor decision.

Nothing in MODIS rises to the bar of the VIIRS F1 fix (where Atlantis was emitting a wrong
permanent-water mask). This notebook now serves as a regression sentinel: if the docstring
loses the "misnomer" wording, or if the LANCE legend ever changes shape, the findings cell will
switch the `F4 [RESOLVED]` line back to `F4 [OPEN]` on next run.